In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
ROOT = "/content/drive/MyDrive/archive"
DATA = ROOT

In [3]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from tqdm import tqdm
import gc
import warnings

# --- Configuration ---
warnings.filterwarnings('ignore')
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 Running on device: {device}")

# Paths (Update these if needed)
ROOT = "/content/drive/MyDrive/archive"
EVENTS_PATH = f"{ROOT}/events.csv"
PROP1_PATH = f"{ROOT}/item_properties_part1.csv"
PROP2_PATH = f"{ROOT}/item_properties_part2.csv"

🚀 Running on device: cuda


In [8]:
# ==========================================
# 1️⃣ Data Loading & Advanced Preprocessing
# ==========================================
print("--- Step 1: Loading & Cleaning Data ---")
events = pd.read_csv(EVENTS_PATH)

# A. Timestamp Correction
if events["timestamp"].max() > 10_000_000_000:
    events["timestamp"] = (events["timestamp"] // 1000).astype(int)

# B. Noise Reduction (CRITICAL)
# Keep items with >= 5 interactions
item_counts = events['itemid'].value_counts()
popular_items = item_counts[item_counts >= 5].index
events = events[events['itemid'].isin(popular_items)].copy()
print(f"Filtered Events: {len(events)} | Unique Items: {events['itemid'].nunique()}")

# C. Event Type Mapping (NEW FEATURE)
# View=1, AddToCart=2, Transaction=3
event_map = {'view': 1, 'addtocart': 2, 'transaction': 3}
events['event_type'] = events['event'].map(event_map).fillna(1).astype(int)

# D. Category Integration (NEW FEATURE - The Game Changer)
print("Loading Item Categories (Optimization: Extracting 'categoryid' only)...")
try:
    # Helper to extract category info without loading full files
    def get_categories(path):
        df = pd.read_csv(path)
        # Property ID '790' is usually the Category ID in RetailRocket
        return df[df['property'] == 'categoryid'][['itemid', 'value']]

    cat1 = get_categories(PROP1_PATH)
    cat2 = get_categories(PROP2_PATH)
    cats = pd.concat([cat1, cat2])
    cats.rename(columns={'value': 'category_id'}, inplace=True)

    # Clean & Convert
    cats['category_id'] = pd.to_numeric(cats['category_id'], errors='coerce').fillna(0).astype(int)
    cats = cats.drop_duplicates(subset='itemid')

    # Merge with Events
    events = events.merge(cats, on='itemid', how='left')
    events['category_id'] = events['category_id'].fillna(0).astype(int)
    print("✅ Categories merged successfully!")
    del cat1, cat2, cats
    gc.collect()
except Exception as e:
    print(f"⚠️ Warning: Could not load categories ({e}). Proceeding without them.")
    events['category_id'] = 0

# E. Sorting & Sessionizing
print("Creating Sessions...")
events = events.sort_values(by=["visitorid", "timestamp"])
events['time_diff'] = events.groupby('visitorid')['timestamp'].diff()
events['new_session'] = (events['visitorid'] != events['visitorid'].shift()) | (events['time_diff'] > 30*60)
events['session_id'] = events['new_session'].cumsum()



--- Step 1: Loading & Cleaning Data ---
Filtered Events: 2491203 | Unique Items: 90948
Loading Item Categories (Optimization: Extracting 'categoryid' only)...
✅ Categories merged successfully!
Creating Sessions...


In [9]:
# ==========================================
# 2️⃣ Mappings & Splitting
# ==========================================
print("--- Step 2: Mappings & Splitting ---")

# Mappings
all_items = events['itemid'].unique()
item2idx = {item: i+1 for i, item in enumerate(all_items)}
n_items = len(item2idx) + 1

all_cats = events['category_id'].unique()
cat2idx = {cat: i+1 for i, cat in enumerate(all_cats)}
n_cats = len(cat2idx) + 1

# Add mapped columns
events['item_idx'] = events['itemid'].map(item2idx)
events['cat_idx'] = events['category_id'].map(cat2idx)

# Chronological Split (7 days Test, 7 days Val)
max_time = events['timestamp'].max()
day_sec = 24 * 60 * 60
test_start = max_time - (7 * day_sec)
val_start = test_start - (7 * day_sec)

train_df = events[events['timestamp'] < val_start]
val_df = events[(events['timestamp'] >= val_start) & (events['timestamp'] < test_start)]
test_df = events[events['timestamp'] >= test_start]

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")



--- Step 2: Mappings & Splitting ---
Train: 2270847 | Val: 117771 | Test: 102585


In [10]:
# ==========================================
# 3️⃣ Data Augmentation (Sliding Window)
# ==========================================
def process_sessions(df, augment=False):
    # Group by session
    grouped = df.groupby('session_id')
    dataset = []

    for _, group in tqdm(grouped, desc=f"Processing (Augment={augment})"):
        items = group['item_idx'].tolist()
        cats = group['cat_idx'].tolist()
        evts = group['event_type'].tolist()

        if len(items) < 2: continue

        if augment:
            # Sliding Window: [1,2,3] -> ([1],2), ([1,2],3)
            for i in range(1, len(items)):
                dataset.append({
                    'items': items[:i+1],
                    'cats': cats[:i+1],
                    'events': evts[:i+1]
                })
        else:
            # Normal: Just the full session
            dataset.append({
                'items': items,
                'cats': cats,
                'events': evts
            })
    return dataset

print("Generating Datasets...")
train_data = process_sessions(train_df, augment=True) # 🔥 Augmentation ON
val_data = process_sessions(val_df, augment=False)
test_data = process_sessions(test_df, augment=False)

Generating Datasets...


Processing (Augment=False): 100%|██████████| 67653/67653 [00:05<00:00, 12538.82it/s]


In [11]:
# ==========================================
# 4️⃣ Dataset & DataLoader
# ==========================================
class EnhancedSessionDataset(Dataset):
    def __init__(self, data):
        self.data = data

    def __len__(self): return len(self.data)

    def __getitem__(self, idx):
        row = self.data[idx]
        # Input sequence (all except last)
        seq_items = row['items'][:-1]
        seq_cats = row['cats'][:-1]
        seq_evts = row['events'][:-1]

        target = row['items'][-1]

        # Graph Construction (Session Graph)
        unique_nodes, inverse = np.unique(seq_items, return_inverse=True)
        adj = np.zeros((len(unique_nodes), len(unique_nodes)))
        for i in range(len(inverse)-1):
            adj[inverse[i], inverse[i+1]] = 1

        # Context Info (Last item features)
        last_idx_local = inverse[-1]

        return (
            torch.tensor(adj, dtype=torch.float),
            torch.tensor(unique_nodes, dtype=torch.long),
            torch.tensor(seq_cats, dtype=torch.long), # Full sequence cats
            torch.tensor(seq_evts, dtype=torch.long), # Full sequence events
            torch.tensor(last_idx_local, dtype=torch.long),
            torch.tensor(target, dtype=torch.long)
        )

def collate_fn(batch):
    adjs, nodes, cats, evts, lasts, targets = zip(*batch)
    max_nodes = max(len(n) for n in nodes)
    max_len = max(len(c) for c in cats) # For sequence features

    pad_adjs, pad_nodes, pad_cats, pad_evts, masks = [], [], [], [], []

    for i in range(len(batch)):
        # Pad Adj
        n = adjs[i].shape[0]
        pad_adj = F.pad(adjs[i], (0, max_nodes-n, 0, max_nodes-n))
        pad_adjs.append(pad_adj)

        # Pad Nodes
        pad_node = F.pad(nodes[i], (0, max_nodes-n))
        pad_nodes.append(pad_node)

        # Mask (True for real nodes)
        mask = torch.zeros(max_nodes, dtype=torch.bool)
        mask[:n] = True
        masks.append(mask)

        # Pad Sequences (Cats/Events) - match node length roughly or distinct
        # Here strictly padding to max_nodes for simplicity in GNN mapping
        # Mapping seq features to unique nodes is complex,
        # SIMPLIFICATION: We use Last Item's Category & Event for global context
        # to keep dimension matching easy in this implementation.
        pad_cats.append(cats[i][-1]) # Taking last item's cat
        pad_evts.append(evts[i][-1]) # Taking last item's event type

    return (
        torch.stack(pad_adjs).to(device),
        torch.stack(pad_nodes).to(device),
        torch.tensor(pad_cats).to(device), # Batch x 1
        torch.tensor(pad_evts).to(device), # Batch x 1
        torch.tensor(lasts).to(device),
        torch.stack(masks).to(device),
        torch.tensor(targets).to(device)
    )

train_loader = DataLoader(EnhancedSessionDataset(train_data), batch_size=128, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(EnhancedSessionDataset(val_data), batch_size=128, shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(EnhancedSessionDataset(test_data), batch_size=128, shuffle=False, collate_fn=collate_fn)



In [12]:
# ==========================================
# 5️⃣ Enhanced SR-GNN Model
# ==========================================
class GatedGraphNN(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.W_in = nn.Linear(hidden_dim, hidden_dim)
        self.W_out = nn.Linear(hidden_dim, hidden_dim)
        self.W_r = nn.Linear(hidden_dim*2, hidden_dim)
        self.W_z = nn.Linear(hidden_dim*2, hidden_dim)
        self.W_h = nn.Linear(hidden_dim*2, hidden_dim)

    def forward(self, adj, nodes):
        in_msg = torch.bmm(adj, nodes)
        out_msg = torch.bmm(adj.transpose(1,2), nodes)
        a = self.W_in(in_msg) + self.W_out(out_msg)
        combined = torch.cat([a, nodes], 2)
        r = torch.sigmoid(self.W_r(combined))
        z = torch.sigmoid(self.W_z(combined))
        h_hat = torch.tanh(self.W_h(combined))
        return (1-z)*nodes + z*h_hat

class SRGNN_Ultimate(nn.Module):
    def __init__(self, n_items, n_cats, hidden_dim=256, dropout=0.3):
        super().__init__()
        self.hidden_dim = hidden_dim

        # Embeddings
        self.item_emb = nn.Embedding(n_items, hidden_dim, padding_idx=0)
        self.cat_emb = nn.Embedding(n_cats, hidden_dim, padding_idx=0)   # Feature 1
        self.event_emb = nn.Embedding(4, hidden_dim)                     # Feature 2 (View/Cart/Buy)

        # GNN
        self.gnn = GatedGraphNN(hidden_dim)

        # Attention
        self.att_w1 = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.att_w2 = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.att_v = nn.Linear(hidden_dim, 1, bias=False)

        # Fusion Layers
        self.dropout = nn.Dropout(dropout)
        self.fc1 = nn.Linear(hidden_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, hidden_dim) # For context fusion
        self.output = nn.Linear(hidden_dim, n_items, bias=False)

    def forward(self, adj, nodes, cats, evts, last_idx, masks):
        # 1. Item Embeddings & GNN
        h = self.item_emb(nodes) # B x N x D
        h = self.gnn(adj, h)

        # 2. Local Session Representation (Last Clicked Item)
        last_idx_ex = last_idx.view(-1, 1, 1).expand(-1, -1, self.hidden_dim)
        s_local = h.gather(1, last_idx_ex).squeeze(1) # B x D

        # 🔥 FUSION: Add Category & Event context to Local Representation
        # This tells the model: "User just [Added to Cart] a [Laptop]"
        h_cat = self.cat_emb(cats)
        h_evt = self.event_emb(evts)

        s_local_enhanced = s_local + (h_cat * 0.5) + (h_evt * 0.5) # Weighted sum

        # 3. Global Session Representation (Attention)
        q = self.att_w1(h)
        k = self.att_w2(s_local_enhanced).unsqueeze(1)
        alpha = torch.sigmoid(self.att_v(torch.tanh(q + k)))
        alpha = alpha.masked_fill(~masks.unsqueeze(-1), 0)
        s_global = torch.sum(alpha * h, 1)

        # 4. Final Prediction
        s_final = self.fc1(s_global) + self.fc2(s_local_enhanced)
        s_final = self.dropout(s_final)
        scores = self.output(s_final)
        return scores



In [13]:
# ==========================================
# 6️⃣ Training Loop
# ==========================================
model = SRGNN_Ultimate(n_items, n_cats, hidden_dim=256, dropout=0.4).to(device)
optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)

best_recall = 0
patience = 0
EPOCHS = 20

print("\n🚀 Starting Training (Ultimate Version)...")
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
        adj, nodes, cats, evts, lasts, masks, targets = batch

        optimizer.zero_grad()
        scores = model(adj, nodes, cats, evts, lasts, masks)
        loss = criterion(scores, targets)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"  Loss: {total_loss/len(train_loader):.4f}")

    # Validation
    model.eval()
    hits, ranks = [], []
    with torch.no_grad():
        for batch in val_loader:
            adj, nodes, cats, evts, lasts, masks, targets = batch
            scores = model(adj, nodes, cats, evts, lasts, masks)
            sub_scores = scores.topk(20)[1].cpu().numpy()
            targets = targets.cpu().numpy()

            for score, target in zip(sub_scores, targets):
                if target in score:
                    hits.append(1)
                    ranks.append(1 / (np.where(score == target)[0][0] + 1))
                else:
                    hits.append(0)
                    ranks.append(0)

    val_recall = np.mean(hits) * 100
    val_mrr = np.mean(ranks)
    print(f"  🔎 Val Recall@20: {val_recall:.2f}% | MRR@20: {val_mrr:.4f}")

    scheduler.step(val_recall)

    if val_recall > best_recall:
        best_recall = val_recall
        torch.save(model.state_dict(), "SRGNN_Ultimate.pth")
        patience = 0
        print("  ✅ Model Saved!")
    else:
        patience += 1
        if patience >= 5:
            print("  🛑 Early Stopping")
            break

# Final Test
print("\n🧪 Final Testing...")
model.load_state_dict(torch.load("SRGNN_Ultimate.pth"))
model.eval()
hits = []
with torch.no_grad():
    for batch in tqdm(test_loader):
        adj, nodes, cats, evts, lasts, masks, targets = batch
        scores = model(adj, nodes, cats, evts, lasts, masks)
        sub_scores = scores.topk(20)[1].cpu().numpy()
        targets = targets.cpu().numpy()
        for score, target in zip(sub_scores, targets):
            hits.append(1 if target in score else 0)

print(f"🏆 Final Test Recall@20: {np.mean(hits)*100:.2f}%")


🚀 Starting Training (Ultimate Version)...


Epoch 1: 100%|██████████| 6621/6621 [05:44<00:00, 19.24it/s]


  Loss: 8.3273
  🔎 Val Recall@20: 41.28% | MRR@20: 0.2222
  ✅ Model Saved!


Epoch 2: 100%|██████████| 6621/6621 [05:42<00:00, 19.31it/s]


  Loss: 6.7890
  🔎 Val Recall@20: 47.25% | MRR@20: 0.2699
  ✅ Model Saved!


...

  Loss: 4.0219
  🔎 Val Recall@20: 53.43% | MRR@20: 0.3251


Epoch 20: 100%|██████████| 6621/6621 [05:41<00:00, 19.41it/s]


  Loss: 3.9728
  🔎 Val Recall@20: 53.20% | MRR@20: 0.3258
  🛑 Early Stopping

🧪 Final Testing...


100%|██████████| 113/113 [00:02<00:00, 53.16it/s]

🏆 Final Test Recall@20: 51.39%
